In [1]:
import h5py
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import vbn_utils
import decoding_utils as du
from analysis_utils import exponential_convolve
import nwb_session_utils as nwb

%matplotlib inline

/opt/anaconda3/envs/vbn_manuscript/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data loading

In [2]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"

In [3]:
units = pd.read_csv(unit_table_file)
units['cortical_layer'] = units['cortical_layer'].replace('3-Feb','2/3') # 2/3 sometimes gets incorrectly reformatted as a date

stim_table = pd.read_csv(stim_table_file)
stim_table = stim_table.drop(columns='Unnamed: 0')

active_tensor = h5py.File(active_tensor_file)

/var/folders/7s/q3zz_qj910x07vwrdqkcgczr0000gp/T/ipykernel_34779/2052873693.py:1: DtypeWarning: Columns (120,123,126,129,132,135,138,141,144,147,150,153,156,159,162,165,174,177,180,183,192,195,198,201,228,229,236) have mixed types. Specify dtype option on import or set low_memory=False.
  units = pd.read_csv(unit_table_file)
/var/folders/7s/q3zz_qj910x07vwrdqkcgczr0000gp/T/ipykernel_34779/2052873693.py:4: DtypeWarning: Columns (32,34,35,43,45) have mixed types. Specify dtype option on import or set low_memory=False.
  stim_table = pd.read_csv(stim_table_file)


In [ ]:
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/ccf_structure_tree_2017.csv")

## Lick-triggered averages

### Compute lick-triggered responses (if pre-computed, load below)

In [ ]:
unit_filter = du.apply_unit_quality_filter(units)
unit_ids = units.loc[unit_filter]['unit_id'].values 
session_list = list(active_tensor.keys())

lick_aligned_unit_summary = {u:[] for u in unit_ids}
stim_filter = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'lickbout_for_flash_during_response_window']
unit_data, shuffle_data, unitIds = vbn_utils.unit_averaged_psth_lick_aligned(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=750, 
                                    resp_window_length=1500)
    

In [ ]:
for udata, sdata, uids in zip(unit_data, shuffle_data, unitIds):
    if len(uids)>0:
        for ud, sd, uid in zip(udata, sdata, uids):
            umean = exponential_convolve(ud, 3, symmetrical=True)
            time_above = np.convolve(umean>sd[2], np.ones(10)).max()
            time_below = np.convolve(umean<sd[1], np.ones(10)).max()
            passes = np.max((time_above, time_below))>9
            lick_aligned_unit_summary[uid] = {'mean': umean, 
                                              'shuffle_mean': sd[0],
                                              'shuffle_ci_low': sd[1],
                                              'shuffle_ci_high': sd[2],
                                              'pass': passes}

In [ ]:
import pickle
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/lick_aligned_unit_data.pkl", 'wb') as file:
    pickle.dump(lick_aligned_unit_summary, file)

### Load pre-computed lick-triggered data

In [ ]:
import pickle
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/lick_aligned_unit_data.pkl", 'rb') as file:
    lick_aligned_unit_summary = pickle.load(file)

In [ ]:
lick_aligned_df = pd.DataFrame.from_dict(lick_aligned_unit_summary, orient='index')

In [ ]:
lick_aligned_df = lick_aligned_df.merge(units[['unit_id', 'cluster_labels_new', 'structure_acronym', 'cortical_layer']], left_index=True, right_on='unit_id')

## Run-start triggered averages

### Compute run-start triggered responses (if pre-computed, load below)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
session_list = units[units['no_anomalies']]['ecephys_session_id'].unique()
acceleration_peths, deceleration_peths, unitIDs = nwb.unit_averaged_psth_time_aligned(session_list, alignment_time_func = 'running', 
                                    time_before=0.5, time_after=0.5, binsize=0.001)

In [ ]:
running_df = {}
for apeths, dpeths, uids in zip(acceleration_peths, deceleration_peths, unitIDs):
    if len(uids)>0:
        for apeth, dpeth, uid in zip(apeths, dpeths, uids):
            running_df[uid] = {'acceleration': apeth, 'deceleration':dpeth}

In [ ]:
import pickle
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/run_start_aligned_unit_data.pkl", 'wb') as file:
    pickle.dump(running_df, file)

### Load pre-computed run-start triggered data

In [ ]:
import pickle
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/run_start_aligned_unit_data.pkl", 'rb') as file:
    running_df = pickle.load(file)

In [ ]:
running_aligned_df = pd.DataFrame.from_dict(running_df, orient='index')
running_aligned_df = running_aligned_df.merge(units[['unit_id', 'cluster_labels_new', 'structure_acronym', 'cortical_layer']], left_index=True, right_on='unit_id')
running_aligned_df.set_index('unit_id', inplace=True)

## Figure panels

In [ ]:
plt.rcParams['figure.dpi'] = 300

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.transforms as transforms
import warnings
warnings.filterwarnings("ignore")

plt.rcParams['font.size'] = 14

def add_vertical_size_bar(ax, size, label, loc,
                           pad=0, borderpad=0.1, sep=2, 
                           prop=None, barcolor="black",
                            **kwargs):
    """Adds a vertical scale bar to the axes."""
    
    trans = transforms.Affine2D().rotate_deg(90) + ax.transData
    size_bar = AnchoredSizeBar(trans,
                             size, label, loc, 
                             pad=pad,
                             color='black',
                             frameon=False,
                             size_vertical=0,
                             )
    ax.add_artist(size_bar)


base_sub = False
session_list = list(active_tensor.keys())
plot_window = slice(0, 500)
time = np.arange(-50, 750)[plot_window]

clusters_to_plot = np.arange(1,9)
for cluster in clusters_to_plot:

    unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & \
                        du.apply_unit_quality_filter(units) 
    unit_ids = units.loc[unit_filter]['unit_id'].values
    num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size

    fig, ax = plt.subplots(1,3)
    fig.set_size_inches([7, 2.25])

    
    # Lick/No-lick stim-aligned comparison
    stim_filters = (
                ['engaged', '~is_change', '~omitted', '~previous_omitted', 
                'flashes_since_change>5', 'lickbout_for_flash_during_response_window', '~is_shared'],
                
                ['engaged', 'is_change', '~omitted', '~previous_omitted', 
                 '~lickbout_for_flash_during_response_window', '~is_shared'],

                ['engaged', 'is_change', '~omitted', '~previous_omitted', 
                 'lickbout_for_flash_during_response_window', '~is_shared'],
                    )
    colors = ['k', 'coral', 'g'] 
    alphas = [0.5, 1, 1] 
    for stim_filter, color, alpha in zip(stim_filters, colors, alphas):
        unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=50, 
                                    resp_window_length=750)
        concat = np.concatenate(unit_data)
        concat = np.array([exponential_convolve(c, 3, symmetrical=True) for c in concat])
        if concat.size==0:
            continue
        
        if base_sub:
            concat = concat - concat[:, :50].mean(axis=1)[:, None]

        vbn_utils.mean_sem_plot(concat[:, plot_window]*1000, ax[0], time, color=color, alpha=alpha)


    # Lick-aligned comparison
    cluster_lick_aligned = lick_aligned_df[(lick_aligned_df['unit_id'].isin(unit_ids))]
    means = np.array([c['mean'] for ic, c in cluster_lick_aligned.iterrows()])
    shuffle_means = np.array([c['shuffle_mean'] for ic, c in cluster_lick_aligned.iterrows()])
    vbn_utils.mean_sem_plot(means*1000, ax[1], np.arange(-500,500), color='k')
    vbn_utils.mean_sem_plot(shuffle_means*1000, ax[1], np.arange(-500,500), color='gray', ls='dotted')


    # Run aligned comparison
    colors=['k', 'gray']
    for ic, condition in enumerate(['acceleration', 'deceleration']):
        means = running_aligned_df.loc[unit_ids][condition].values
        means = np.array([exponential_convolve(m, 3, True) for m in means])
        vbn_utils.mean_sem_plot(means, ax[2], np.arange(-500,500), color=colors[ic])
        num_nonan = np.sum([~np.isnan(m[0]) for m in means])
    print(f'cluster {cluster} count: {len(unit_ids)}; run aligned count: {num_nonan}')
    
    ymin = np.min([a.get_ylim()[0] for a in ax])
    ymax = np.max([a.get_ylim()[1] for a in ax])

    yminint = int(np.ceil(ymin))
    a = ax[0]
    a.spines['left'].set_bounds(yminint, yminint+5)
    a.set_yticks([yminint, yminint+5])
    a.spines['bottom'].set_bounds(0, 250)
    a.set_xticks([0,250])
    a.spines['top'].set_visible(False)
    a.spines['right'].set_visible(False)
    a.tick_params(labelleft=False,)

    [a.set_ylim(ymin, ymax) for a in ax]
    [a.axis('off') for a in ax[1:]]
    [a.axvline(0, color='k', ls='dotted') for a in ax[1:]]


    plt.tight_layout()
    

## GLM dropout analysis

In [ ]:
dropouts = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/GLM_dropout_with_unit_info_active_passive.csv")

In [ ]:
for region in ['VISall', 'VISp', ['LGd', 'LP'], 'SCMRN', 'Hipp']:
    good_units = vbn_utils.get_unit_ids(dropouts, region)
    region_dropouts = dropouts.set_index('unit_id').loc[good_units]

    print(region)
    print(region_dropouts["('variance_explained_full', 'Full')"].describe())

In [ ]:
metric = 'absolute_change_from_full'
for region in ['VISall', 'SCMRN']:
    good_units = vbn_utils.get_unit_ids(dropouts, region)
    region_dropouts = dropouts.set_index('unit_id').loc[good_units]

    print(region, 'images', np.sum(region_dropouts[f"('{metric}', 'all-images')"].values<-0.01)/len(region_dropouts), len(region_dropouts))
    print(region, 'licks', np.sum(region_dropouts[f"('{metric}', 'licks')"].values<-0.01)/len(region_dropouts))

In [ ]:
columns_to_plot = [f"('{metric}', 'all-images')",
                   f"('{metric}', 'omissions')",
                   f"('{metric}', 'task')",
                   f"('{metric}', 'pupil')",
                   f"('{metric}', 'licks')",
                   f"('{metric}', 'running')"
                   ]

In [ ]:
pt = dropouts[dropouts['cluster_labels_new'].isin(np.arange(9))].pivot_table(index='cluster_labels_new', values=columns_to_plot, aggfunc='mean')

In [ ]:
pt = pt[columns_to_plot]

In [ ]:
pt_col_normed = pt.div(pt.sum(axis=0), axis=1)
pt_row_normed = pt.div(pt.sum(axis=1), axis=0)

In [ ]:
from matplotlib.pyplot import xticks

toplot = pt_col_normed

plt.rcParams.update({'font.size': 14})
plt.figure(figsize=(5,5))
plt.imshow(toplot.values, aspect='equal', cmap='Greys_r')

plt.xticks(np.arange(toplot.shape[1]), [c.split(',')[1].replace(')', '').replace("'", '') for c in toplot.columns], rotation=90)
plt.yticks(np.arange(toplot.shape[0]), toplot.index.astype(int))
plt.ylabel('Cluster')
plt.xlabel('Kernel')
cbar = plt.colorbar()
cbar.set_label('Norm dropout score')